# Creating Olist Source Codes

This notebook now treats `customer_unique_id` as the real customer key. It creates a deduplicated source table with one row per real Olist customer, then fans that canonical source code back out to the original `customer_id` rows for order-level analysis.


In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "rebuild_unique_customer_mapping.py").exists():
            return candidate
    raise FileNotFoundError("Could not find rebuild_unique_customer_mapping.py from the current notebook path.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.options.display.max_columns = 120

from rebuild_unique_customer_mapping import (
    OLIST_CUSTOMERS_WITH_SOURCE_CSV,
    OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV,
    SOURCE_CODE_SUMMARY_CSV,
    build_source_outputs,
    build_unique_customers,
    load_olist_customers,
)


In [ ]:
olist_customers_df = load_olist_customers()

print(f"Loaded Olist customer-order rows: {len(olist_customers_df):,}")
print(f"Unique customer_unique_id values: {olist_customers_df['customer_unique_id'].nunique():,}")
display(olist_customers_df.drop(columns=["_row_order"]).head())


In [ ]:
unique_customers_df = build_unique_customers(olist_customers_df)

print(f"Unique customer rows: {len(unique_customers_df):,}")
print(f"Repeat customers represented: {(unique_customers_df['order_count'] > 1).sum():,}")
display(unique_customers_df.head())


In [ ]:
source_code_summary_df, customer_orders_with_source_df = build_source_outputs(
    olist_customers_df,
    unique_customers_df,
)

print(f"Unique source codes: {len(source_code_summary_df):,}")
print(f"Source-code customer total: {source_code_summary_df['count'].sum():,}")
display(source_code_summary_df.head(20))


In [ ]:
unique_customers_df.to_csv(OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV, index=False)
customer_orders_with_source_df.to_csv(OLIST_CUSTOMERS_WITH_SOURCE_CSV, index=False)
source_code_summary_df.to_csv(SOURCE_CODE_SUMMARY_CSV, index=False)

print(f"Wrote unique customers with source codes: {OLIST_UNIQUE_CUSTOMERS_WITH_SOURCE_CSV}")
print(f"Wrote order rows with canonical source codes: {OLIST_CUSTOMERS_WITH_SOURCE_CSV}")
print(f"Wrote source-code summary: {SOURCE_CODE_SUMMARY_CSV}")


In [ ]:
validation = pd.DataFrame(
    {
        "check": [
            "raw customer-order rows",
            "unique customer rows",
            "source summary total",
            "order rows with source codes",
            "customer_unique_id with multiple canonical source_codes",
        ],
        "value": [
            len(olist_customers_df),
            len(unique_customers_df),
            int(source_code_summary_df["count"].sum()),
            len(customer_orders_with_source_df),
            int((unique_customers_df.groupby("customer_unique_id")["source_code"].nunique(dropna=False) > 1).sum()),
        ],
    }
)

display(validation)
